In [ ]:
# https://github.com/huggingface/sentence-transformers/blob/8024980cf0c4b4bb977041cafd2aa90ef7b3c560/sentence_transformers/sparse_encoder/model.py#L35
# https://github.com/huggingface/sentence-transformers/blob/328398d22050ac64737f2400c9283041b08aa59a/sentence_transformers/sparse_encoder/modules/splade_pooling.py#L14
# https://github.com/huggingface/sentence-transformers/blob/328398d22050ac64737f2400c9283041b08aa59a/sentence_transformers/base/model.py#L47

In [2]:
import torch 

torch.tensor([
  [0.0, 0.0, 1.7, 0.0, 0.0, 2.3],
  [0.0, 5.1, 0.0, 0.0, 0.0, 0.0]
]).to_sparse()

tensor(indices=tensor([[0, 0, 1],
                       [2, 5, 1]]),
       values=tensor([1.7000, 2.3000, 5.1000]),
       size=(2, 6), nnz=3, layout=torch.sparse_coo)

In [ ]:
import torch
from sentence_transformers.sparse_encoder.modules import SpladePooling
from sentence_transformers.base.modality_types import TextInput

# TextInput: str

pool = SpladePooling(pooling_strategy="max", activation_function="relu")
pool.eval()  # important: inference mode

features = {
    "token_embeddings": torch.tensor([
        [  # batch item 0
            [ 1.0, -1.0,  0.0, 2.0],   # token 1 logits
            [ 0.5,  3.0, -2.0, 1.0],   # token 2 logits
            [10.0, 10.0, 10.0, 10.0],  # token 3 logits, but masked out below
        ]
    ]),
    "attention_mask": torch.tensor([
        [1, 1, 0]
    ])
}

out = pool(features)
print(out["sentence_embedding"])

/home/lpozzi/Git/data-science-lectures/lm_ir/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tensor([[0.6931, 1.3863, 0.0000, 1.0986]])


In [ ]:
# token_embeddings has shape:

# (1, 3, 4)

# meaning:

# 1 sentence in the batch
# 3 token positions
# for each token position, 4 vocabulary scores

In [4]:
from sentence_transformers import SparseEncoder
model = SparseEncoder("jhu-clsp/mmBERT-base", device='cpu')

Loading weights: 100%|██████████| 138/138 [00:00<00:00, 63438.62it/s]


In [8]:
features = model.preprocess(['hello words', 'hello worlds'])
embeddings = model(features)

In [9]:
embeddings

{'input_ids': tensor([[    2, 25612,  3907,     1],
        [    2, 25612, 33101,     1]]), 'attention_mask': tensor([[1, 1, 1, 1],
        [1, 1, 1, 1]]), 'modality': 'text', 'token_embeddings': tensor([[[-11.3898,  13.9980,  28.3794,  ...,  -8.3486, -11.6033, -11.3903],
         [ -7.9173,   3.4510,   1.5991,  ...,  -7.9225, -10.4187,  -7.9177],
         [ -8.7832,   4.5649,   1.5557,  ...,  -8.5460, -10.0934,  -8.7835],
         [ -4.2157,  29.5041,  16.2933,  ...,  -6.7432, -12.1348,  -4.2161]],

        [[-11.2810,  14.2041,  28.2143,  ...,  -8.3503, -11.6653, -11.2814],
         [ -7.8732,   3.7415,   2.0140,  ...,  -8.7015, -10.7180,  -7.8736],
         [ -8.5239,   0.8679,  -0.0846,  ...,  -8.4124, -11.7852,  -8.5243],
         [ -4.0491,  27.6203,  15.3372,  ...,  -7.5606, -14.3153,  -4.0495]]],
       grad_fn=<ViewBackward0>), 'sentence_embedding': tensor([[0.0000, 3.4179, 3.3803,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 3.3541, 3.3747,  ..., 0.0000, 0.0000, 0.0000]],


Yes. The math of **`SpladePooling`** is:

For each token position, take the **MLM logits over the vocabulary**, apply a nonlinearity, then **pool across the sequence dimension**.

If the input logits are

[
L \in \mathbb{R}^{B \times T \times V}
]

with:

* **(B)** = batch size
* **(T)** = sequence length
* **(V)** = vocabulary size

and the attention mask is

[
M \in {0,1}^{B \times T}
]

then `SpladePooling` computes a sentence vector

[
S \in \mathbb{R}^{B \times V}
]

---

## Step 1: mask padded tokens

The mask is expanded to shape ((B, T, 1)) and multiplied elementwise:

[
\tilde{L}*{b,t,v} = L*{b,t,v} \cdot M_{b,t}
]

So if token position (t) is masked out, **all vocab scores at that position become 0**.

---

## Step 2: apply ReLU

Negative logits are removed:

[
R_{b,t,v} = \max(\tilde{L}_{b,t,v}, 0)
]

So only positive evidence survives.

---

## Step 3: apply `log1p`

In the normal `"relu"` mode, the transformed score is:

[
Z_{b,t,v} = \log(1 + R_{b,t,v})
]

This compresses large values while keeping zeros at zero.

If `activation_function="log1p_relu"`, then it applies another `log1p`:

[
Z_{b,t,v} = \log\big(1 + \log(1 + R_{b,t,v})\big)
]

So that variant compresses large values even more.

---

## Step 4: pool across tokens

Now you have a transformed tensor (Z) of shape ((B,T,V)).
You collapse the token dimension (T) to get one vector per sentence.

### Max pooling

If `pooling_strategy="max"`:

[
S_{b,v} = \max_{t=1,\dots,T} Z_{b,t,v}
]

This means each vocabulary term keeps its **strongest activation** anywhere in the sequence.

### Sum pooling

If `pooling_strategy="sum"`:

[
S_{b,v} = \sum_{t=1}^{T} Z_{b,t,v}
]

This means each vocabulary term accumulates contributions from **all token positions**.

---

# Compact formula

For the default setup, the whole thing is:

### Max version

[
S_{b,v} = \max_t \log\big(1 + \max(L_{b,t,v} \cdot M_{b,t}, 0)\big)
]

### Sum version

[
S_{b,v} = \sum_t \log\big(1 + \max(L_{b,t,v} \cdot M_{b,t}, 0)\big)
]

And for `"log1p_relu"` you wrap one more `log(1 + ·)` around the transformed term.

---

# Tiny numeric example

Take one example with:

* batch size = 1
* sequence length = 2
* vocab size = 4

Logits:

[
L =
\begin{bmatrix}
[
[1.0,\ -1.0,\ 0.0,\ 2.0], \
[0.5,\ 3.0,\ -2.0,\ 1.0]
]
\end{bmatrix}
]

Mask:

[
M =
\begin{bmatrix}
[1,\ 1]
\end{bmatrix}
]

---

## After masking

No change here, since both tokens are valid.

[
\tilde{L} =
\begin{bmatrix}
[
[1.0,\ -1.0,\ 0.0,\ 2.0], \
[0.5,\ 3.0,\ -2.0,\ 1.0]
]
\end{bmatrix}
]

---

## After ReLU

[
R =
\begin{bmatrix}
[
[1.0,\ 0.0,\ 0.0,\ 2.0], \
[0.5,\ 3.0,\ 0.0,\ 1.0]
]
\end{bmatrix}
]

---

## After `log1p`

Use:

* (\log(1+1.0)=0.6931)
* (\log(1+0.5)=0.4055)
* (\log(1+3.0)=1.3863)
* (\log(1+2.0)=1.0986)
* (\log(1+0)=0)

So:

[
Z =
\begin{bmatrix}
[
[0.6931,\ 0,\ 0,\ 1.0986], \
[0.4055,\ 1.3863,\ 0,\ 0.6931]
]
\end{bmatrix}
]

---

## Max pooling

Take max across the token rows:

[
S =
[0.6931,\ 1.3863,\ 0,\ 1.0986]
]

---

## Sum pooling

Add the token rows:

[
S =
[1.0986,\ 1.3863,\ 0,\ 1.7917]
]

---

# Why this creates sparse vectors

Because:

* negative logits are killed by **ReLU**
* masked tokens are zeroed out
* many vocab terms never get positive evidence
* so many dimensions stay exactly **0**

Then after pooling, the final sentence vector has shape:

[
(B,V)
]

but most coordinates are zero, which is why it is useful as a sparse representation.

---

# How to map this to the code

This block:

```python id="403jm9"
masked_current_chunk_logits = current_chunk_logits * current_chunk_mask

current_chunk_transformed = masked_current_chunk_logits.relu_()
current_chunk_transformed = current_chunk_transformed.log1p_()
```

is exactly:

[
\log(1 + \max(L \cdot M, 0))
]

Then:

```python id="k7m4qd"
chunk_pooled = torch.max(current_chunk_transformed, dim=1)[0]
```

is:

[
\max_t Z_{b,t,v}
]

and:

```python id="s6nbo9"
chunk_pooled = torch.sum(current_chunk_transformed, dim=1)
```

is:

[
\sum_t Z_{b,t,v}
]

---

# One-sentence summary

**SpladePooling turns per-token full-vocabulary MLM logits into one sparse sentence vector by applying `ReLU`, then `log1p`, then pooling across token positions with either `max` or `sum`.**

I can also rewrite it as a concrete worked example using your exact toy tensor line by line.
